In [274]:
#%load_ext autoreload
#%autoreload 2

 
import sys, pathlib, json, pprint, pandas as pd 
from pathlib import Path
from pydantic import Field,BaseModel 
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

from runtime.v4.semantics.load_semantics import load_semantics, load_idiom_rules
from runtime.v4.semantics.semantic_models import * 
from runtime.v4.analyst_agent.catalog import Catalog
from runtime.v4.analyst_agent.smart_data import SmartData
from runtime.v4.analyst_agent.smart_data_tools  import SmartDataTools
from runtime.v4.analyst_agent.analyst_models  import *
from runtime.v4.analyst_agent.analyst_prompts import system_prompt_template3 as agent_system_prompt


import  os  
from datetime import datetime 
from langchain_core.tools import StructuredTool, Tool


import duckdb
from typing import Any, Dict, List, Iterable, Union, Optional 
import yaml
import pprint
import pandas as pd, numpy as np
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig, Runnable, RunnableLambda
 
from langchain.tools import tool, ToolRuntime
from langgraph.runtime import get_runtime 
from langchain.agents import create_agent
from langchain_core.runnables import RunnableLambda
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_core.runnables import Runnable
from langchain.tools import tool, ToolRuntime
from langgraph.runtime import get_runtime 
from langchain.agents import create_agent
 

In [275]:

#inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
#pinj = pd.read_csv("../datasets/IX5I_4P/producers.csv")
#locs= pd.read_csv("../datasets/IX5I_4P/locations.csv")

inj = pd.read_csv("../datasets/Demo1/injectors.csv")
pinj = pd.read_csv("../datasets/Demo1/producers.csv")
locs= pd.read_csv("../datasets/Demo1/locations.csv")



inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
inj['DAY']   = inj['DATE'].dt.day
inj['MONTH'] = inj['DATE'].dt.month
inj['YEAR']  = inj['DATE'].dt.year
pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
pinj['DAY']   = pinj['DATE'].dt.day
pinj['MONTH'] = pinj['DATE'].dt.month
pinj['YEAR']  = pinj['DATE'].dt.year
 
from get_llm_model import azure_llm_if
from support import * 

In [276]:

semantics_json = Path('../runtime/v4/semantics/semantic_models.json')
semantic_catalog = load_semantics(semantics_json)
known_table_models = { t.name: t for t in semantic_catalog.tables } 

df_dict = {'injectors': inj, 'producers':pinj , 'locations': locs }
data = SmartData()
data.initialize_from_named_dataframes( df_dict, known_table_models)

smart_data_tools = SmartDataTools( data=data)
tools = smart_data_tools.get_tools()
print(smart_data_tools.catalog_snapshot())


idiom_rules_path = Path('../runtime/v4/semantics/idioms.json')
idiom_rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = idiom_rules_path )
idiom_rules

------------------------------------------------------------

Table: injectors
  description: Injector water injection time series.
  kind: base
  row_count: 20850
  Columns:
    - name: DATE | data_type: timestamp | description: Injection date. | derived_column: False
    - name: NAME | data_type: string | description: Injector well identifier. | derived_column: False
    - name: WATER_INJECTION_VOLUME | data_type: float | description: Injected water volume. | derived_column: False
    - name: SUBZONE | data_type: string | description: Vertical subzone. | derived_column: False
    - name: SECTOR | data_type: integer | description: Geographic sector. | derived_column: False
    - name: YEAR | data_type: integer | description: Year from DATE. | derived_column: False
    - name: MONTH | data_type: integer | description: Month from DATE. | derived_column: False
    - name: DAY | data_type: integer | description: Day from DATE. | derived_column: False

Table: producers
  description: Produ

{'date subtraction': "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
 'date truncation': "Use DATE_TRUNC('month', column).",
 'reserved keywords': 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
 'string concatenation': 'Use the || operator or CONCAT().',
 'boolean aggregation': 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.',
 'nested aggregates': 'Avoid nested aggregates—never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)',
 'cte helpers': 'Use CTEs to capture helper scalars (like current_year via MAX("DATE")) before performing group aggregations. For example:\n    WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ..

In [277]:
constraints = "".join([f"- {i}\n" for i in semantic_catalog.semantic_constraints])
print(constraints)

idiom_context = "".join([f"- {i}: {v}\n" for i,v in idiom_rules.items()])
print( idiom_context )



- WELL_TYPE domain is restricted to exactly two values: 'Injector' and 'Producer'.
- Each well belongs to exactly one WELL_TYPE category (mutually exclusive).
- Well identifiers are stored in NAME and used consistently as join keys across tables.
- injectors.NAME joins to locations.NAME only for rows where locations.WELL_TYPE = 'Injector'.
- producers.NAME joins to locations.NAME only for rows where locations.WELL_TYPE = 'Producer'.
- All volume measures are non-negative (WATER_INJECTION_VOLUME, LIQUID_VOLUME, WATER_VOLUME, OIL_VOLUME, GAS_VOLUME).
- Production balance constraint: LIQUID_VOLUME is approximately WATER_VOLUME + OIL_VOLUME + GAS_VOLUME (allowing small numerical tolerance).
- If YEAR, MONTH, DAY are present, they should match the corresponding DATE components.

- date subtraction: Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().
- date truncation: Use DATE_TRUNC('month', column).
- reserved keywords: Always wrap the column name "DATE" in double quo

# Direct ReAct agent. Simple

In [456]:

single_agent_data = SmartData()
single_agent_data.initialize_from_named_dataframes( df_dict, known_table_models )
single_agnet_tools = SmartDataTools( single_agent_data ).get_tools()

#from runtime.v4.analyst_agent.analyst_prompts import system_prompt_template3 as system_prompt_template3
from runtime.v4.analyst_agent.analyst_prompts import system_prompt_template1b as system_prompt_template1
#prompt = system_prompt_template3.format( idiom = 'duckdb', idiom_examples = idiom_context, constraints = constraints)
prompt = system_prompt_template1.format( idiom = 'duckdb', idiom_examples = idiom_context, constraints = constraints)

prompt = prompt + "\n\nALWAYS produce a final table with all the information needed for charting without need of further processing"
print( prompt )




You are an expert analyst of databases.  
Your job is answer user questions grounded in the information contained in the database

Workflow:
You must:
1. Always call catalog_snapshot first.

2. Analyze the question and the information in the catalog and produce a concise PLAN
The PLAN must be concise and must include:
- required source tables
- whether existing derived tables can be reused
- target table names to materialize
- high-level transformation logic, without SQL


3. You MUST ALWAYS record the PLAN in plain text. Only after the PLAN message is sent may you call sql_* tools.
4. Use sql_materialize to create intermediate tables.
5. When multiple output tables are to be produced, proceed sequentially one at a time  
6. Your job finishes once all the target tables are confirmed present (either via initial audit or your materializations).  

Important:
- The name of generated tables and columns should reflect the table contents  

- Use lowercase snake_case for table names and col

In [457]:

llm = azure_llm_if()
print( llm )
single_agent_data.clear_derived()
agent = create_agent(
        model=llm,
        system_prompt=prompt,
        tools=single_agnet_tools,
        response_format=AgentTableResponse,
        #checkpointer= MemorySaver() 
    )


#name = response['structured_response'].tables[0].table_name
#data.get_table_as_df( name )

client=<openai.resources.chat.completions.completions.Completions object at 0x0000029AF3435F90> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000029AF3A0E260> root_client=<openai.lib.azure.AzureOpenAI object at 0x0000029AF3A80820> root_async_client=<openai.lib.azure.AsyncAzureOpenAI object at 0x0000029AF34366B0> model_name='gpt-4o' temperature=0.0 model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True azure_endpoint='https://openai-if-test.openai.azure.com/' deployment_name='gpt-4' openai_api_version='2024-12-01-preview' openai_api_type='azure'


In [458]:
query3_1 = "Tell me the total water injection volume for each quater since 2018 split by subzone and yer"
messages = {"messages": [{"role": "user", "content": query3_1}]}
response = run_agent_stream_values(agent, messages )



--- HumanMessage ---
name: None
Tell me the total water injection volume for each quater since 2018 split by subzone and yer

--- AIMessage ---
name: None
TOOL CALL: catalog_snapshot
ARGS: {}

--- ToolMessage ---
name: catalog_snapshot
------------------------------------------------------------

Table: injectors
  description: Injector water injection time series.
  kind: base
  row_count: 20850
  Columns:
    - name: DATE | data_type: timestamp | description: Injection date. | derived_column: False
    - name: NAME | data_type: string | description: Injector well identifier. | derived_column: False
    - name: WATER_INJECTION_VOLUME | data_type: float | description: Injected water volume. | derived_column: False
    - name: SUBZONE | data_type: string | description: Vertical subzone. | derived_column: False
    - name: SECTOR | data_type: integer | description: Geographic sector. | derived_column: False
    - name: YEAR | data_type: integer | description: Year from DATE. | derived_c

In [459]:
response['structured_response']


AgentTableResponse(agent='analyst', user_query='Tell me the total water injection volume for each quarter since 2018 split by subzone and year.', tables=[TableItemAgentResponse(table_name='quarterly_water_injection_by_subzone_year', description='This table contains the total water injection volume for each quarter since 2018, grouped by year, quarter, and subzone. Columns include year, quarter, subzone, and total_water_injection_volume.')])

In [460]:
from __future__ import annotations

from typing import Any, Iterable
import pandas as pd

import copy
import json
import pandas as pd


PLOTLY_ARRAY_KEYS = {
    "x",
    "y",
    "z",
    "r",
    "theta",
    "values",
    "labels",
    "text",
    "hovertext",
    "customdata",
    "ids",
    "parents",
}


def resolve_plotly_figure(
    figure: dict,
    df: pd.DataFrame,
    *,
    to_json: bool = False,
) -> dict | str:
    """
    Replace column-name placeholders in Plotly traces with real DataFrame values.

    Example:
        "x": "DATE"              -> "x": [...]
        "y": "oil_production"    -> "y": [...]
        "labels": "producer"     -> "labels": [...]
        "values": "oil_rate"     -> "values": [...]
    """

    resolved = copy.deepcopy(figure)

    for trace in resolved.get("data", []):
        for key in PLOTLY_ARRAY_KEYS:
            value = trace.get(key)

            if isinstance(value, str) and value in df.columns:
                trace[key] = df[value].tolist()

    if to_json:
        return json.dumps(resolved, default=str)

    return resolved
class TableResponseProcessor:
    """
    Builds compact, chart-oriented table summaries for a Plotly presenter LLM.

    The LLM receives metadata and limited column examples, but never the full data.
    """

    def __init__(
        self,
        max_examples: int = 3,
        low_cardinality_threshold: int = 10,
        medium_cardinality_threshold: int = 50,
        categorical_numeric_threshold: int = 12,
    ):
        self.max_examples = max_examples
        self.low_cardinality_threshold = low_cardinality_threshold
        self.medium_cardinality_threshold = medium_cardinality_threshold
        self.categorical_numeric_threshold = categorical_numeric_threshold

    def process(
        self,
        items: Iterable[tuple["TableCard", pd.DataFrame]],
    ) -> str:
        """
        Process multiple (TableCard, DataFrame) pairs into one text block
        for the presenter prompt.
        """

        blocks: list[str] = []

        for table_card, df in items:
            blocks.append(self.process_table(df=df, table_card=table_card))

        #return "\n\n" + ("=" * 80) + "\n\n".join(blocks)
        return ("\n\n" + "=" * 80 + "\n\n").join(blocks)

    def process_table(
        self,
        df: pd.DataFrame,
        table_card: "TableCard",
    ) -> str:
        """
        Convert one DataFrame + TableCard into compact, LLM-friendly text.
        """

        table_name = getattr(table_card, "name", "unknown_table")
        table_description = getattr(table_card, "description", None)

        lines: list[str] = []

        lines.append(f"Table: {table_name}")

        if table_description:
            lines.append(f"Description: {table_description}")

        lines.append(f"Rows: {len(df)}")
        lines.append(f"Columns: {len(df.columns)}")

        lines.append("")
        lines.append("Column summaries:")

        for col in df.columns:
            summary = self.infer_column_summary(df[col])

            description = self.get_column_description(table_card, col)
            if description:
                summary["description"] = description

            lines.append(f"- Column: {col}")

            preferred_order = [
                "description",
                "dtype",
                "role",
                "unique_count",
                "unique_ratio",
                "cardinality",
                "null_count",
                "null_ratio",
                "min",
                "max",
                "mean",
                "median",
                "min_date",
                "max_date",
                "common_interval",
                "is_monotonic_increasing",
                "has_negative_values",
                "has_zero_values",
                "example_values",
            ]

            for key in preferred_order:
                if key in summary:
                    lines.append(f"  {key}: {summary[key]}")

        return "\n".join(lines)

    def infer_column_role(self, s: pd.Series) -> str:
        """
        Infer a chart-oriented semantic role.
        """

        if pd.api.types.is_datetime64_any_dtype(s):
            return "temporal"

        if pd.api.types.is_bool_dtype(s):
            return "categorical"

        if pd.api.types.is_numeric_dtype(s):
            nunique = s.nunique(dropna=True)

            if nunique <= self.categorical_numeric_threshold:
                return "categorical_numeric"

            return "quantitative"

        if pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s):
            parsed = pd.to_datetime(s.dropna(), errors="coerce")
            valid_ratio = parsed.notna().mean() if len(parsed) else 0.0

            if valid_ratio >= 0.9:
                return "temporal"

            return "categorical"

        return "unknown"

    def infer_column_summary(self, s: pd.Series) -> dict[str, Any]:
        """
        Produce compact metadata for one DataFrame column.
        """

        non_null = s.dropna()

        row_count = len(s)
        non_null_count = len(non_null)
        null_count = row_count - non_null_count
        null_ratio = null_count / row_count if row_count else 0.0

        unique_count = non_null.nunique(dropna=True)
        unique_ratio = unique_count / row_count if row_count else 0.0

        role = self.infer_column_role(s)

        summary: dict[str, Any] = {
            "dtype": str(s.dtype),
            "role": role,
            "non_null_count": int(non_null_count),
            "null_count": int(null_count),
            "null_ratio": round(null_ratio, 4),
            "unique_count": int(unique_count),
            "unique_ratio": round(unique_ratio, 4),
        }

        if role in {"categorical", "categorical_numeric"}:
            summary["cardinality"] = self.classify_cardinality(unique_count)

            examples = (
                non_null
                .drop_duplicates()
                .astype(str)
                .head(self.max_examples)
                .tolist()
            )

            if examples:
                summary["example_values"] = examples

        elif role == "quantitative":
            numeric = pd.to_numeric(non_null, errors="coerce").dropna()

            if not numeric.empty:
                summary.update(
                    {
                        "min": round(float(numeric.min()), 4),
                        "max": round(float(numeric.max()), 4),
                        "mean": round(float(numeric.mean()), 4),
                        "median": round(float(numeric.median()), 4),
                        "has_negative_values": bool((numeric < 0).any()),
                        "has_zero_values": bool((numeric == 0).any()),
                    }
                )

        elif role == "temporal":
            dt = pd.to_datetime(non_null, errors="coerce").dropna()

            if not dt.empty:
                summary.update(
                    {
                        "min_date": str(dt.min()),
                        "max_date": str(dt.max()),
                        "is_monotonic_increasing": bool(dt.is_monotonic_increasing),
                    }
                )

                if len(dt) > 1:
                    diffs = dt.sort_values().diff().dropna()
                    if not diffs.empty:
                        mode_diff = diffs.mode()
                        if not mode_diff.empty:
                            summary["common_interval"] = str(mode_diff.iloc[0])

        return summary

    def classify_cardinality(self, unique_count: int) -> str:
        """
        Convert unique count into low/medium/high cardinality label.
        """

        if unique_count <= self.low_cardinality_threshold:
            return "low"

        if unique_count <= self.medium_cardinality_threshold:
            return "medium"

        return "high"

    def get_column_description(
        self,
        table_card: "TableCard",
        column_name: str,
    ) -> str | None:
        """
        Extract column description from a TableCard, if present.

        Supports columns represented as either objects or dictionaries.
        """

        columns = getattr(table_card, "columns", None)

        if not columns:
            return None

        for col in columns:
            if isinstance(col, dict):
                name = col.get("name")
                description = col.get("description")
            else:
                name = getattr(col, "name", None)
                description = getattr(col, "description", None)

            if name == column_name:
                return description

        return None

In [461]:
PLOTLY_PRESENTER_PROMPT1 = """
You are a Plotly.js figure JSON generator.

You are given:
- a user query
- one table summary
- the table name
- column names, column roles, and column descriptions when available

Your job is to return only a Plotly.js figure JSON object that answers the user query.

The output will be used as:

Plotly.newPlot(element, figure.data, figure.layout, figure.config)

USER QUERY
{user_query}

TABLE
{tables}

STRICT OUTPUT RULES
- Return only valid JSON.
- Do not return markdown.
- Do not explain anything.
- Do not include comments.
- Do not include extra top-level fields.
- The top-level JSON object must contain only: data, layout, config.
- Use only valid Plotly figure keys.
- Do not use custom keys like table, column, split_by, placeholder, encoding, or fields.
- Do not use real data values.
- Use only columns that exist in the table summary.
- Do not invent column names.
- Wherever Plotly expects an array of data values, use the column name as a string placeholder.

PLACEHOLDER RULE
Use only the raw column name as the placeholder.

Examples:
"x": "DATE"
"y": "oil_production"
"text": "producer"
"labels": "producer"
"values": "oil_production"

CHART SELECTION RULES
- temporal + quantitative -> scatter lines+markers
- categorical + quantitative -> bar
- ranked/top/bottom -> bar, horizontal if labels are long
- share/percentage/fraction/contribution of total -> pie, only for low-cardinality categories
- two quantitative columns row-by-row -> scatter markers
- one numeric value -> indicator
- distribution/frequency/spread of one numeric column -> histogram
- temporal + several numeric columns in wide table -> one line trace per numeric column
- temporal + category + quantity in long table -> one line trace only; put category in text/hovertext
- several metrics with same unit -> multiple traces
- metrics with different units -> use y2 only if clearly needed
- use heatmap only for x-category + y-category + numeric-value matrix style data
- never use Plotly transforms
- never aggregate, sort, filter, group, or calculate unless already present in the table

STYLE RULES
- Include a clear title.
- Include axis titles when the chart has x/y axes.
- Use hovertemplate when useful.
- Set config.responsive to true.
- Set config.displaylogo to false.

RETURN SHAPE
Return only JSON shaped like this:

{{
  "data": [
    {{
      "type": "scatter",
      "mode": "lines+markers",
      "x": "<column_name>",
      "y": "<column_name>"
    }}
  ],
  "layout": {{
    "title": {{
      "text": "<chart title>"
    }},
  }},
  "config": {{
    "responsive": true,
    "displaylogo": false
  }}
}}
"""

In [ ]:
PLOTLY_PRESENTER_PROMPT2 = """
You are a Plotly.js figure JSON generator.

You are given:
- a user query
- one table summary
- the table name
- column names, column roles, and column descriptions when available

===============================================================================
YOUR RESPONSIBILITIES
===============================================================================
- Analyze the user intent and the information provided in the table. 
- Decide HOW best display the information to a human.
- ALWAYS output your reasoning steps.
- Consider:
    1. What variable or variables must be in the X axis
    2. Determine if the variable in the X axis is Categorical (Nominal, ordinal), numerical or time?
    3. What variable(s) must be in the Y axis
    4. Is this a single-trace plot?
    5. From the user intent, determine if grouping is needed.  
    6. You must print a  Plotly.js figure JSON object that answers the user query.
 

===============================================================================
STRICT plotly JSON RULES
===============================================================================
- enclode the json plotly figure inside <plot> and </plot> 
- Do not include extra top-level fields.
- NO TRANSFORMS: Never use the transforms property.
- GROUPING VIA TRACES: If grouping is needed (e.g., by Subzone), create a separate trace object {{ "type": "bar", "name": "GROUP_NAME", ... } for each unique group.
- The top-level JSON object must contain only: data, layout, config.
- Use only valid Plotly figure keys.
- Do not use custom keys like table, column, split_by, placeholder, encoding, or fields.
- Do not use real data values.
- USE ONLY COLUMNS THAT EXIST in the table summary.
- Do not invent column names.
- Wherever Plotly expects an array of data values, use the column name as a string placeholder.

===============================================================================
PLACEHOLDER RULE
===============================================================================
Use only the raw column name as the placeholder.

Examples:
"x": "DATE"
"y": "oil_production"
"text": "producer"
"labels": "producer"
"values": "oil_production"

    

USER QUERY
{user_query}

TABLE
{tables}

""" 

k = """
===============================================================================
YOUR RESPONSIBILITIES
===============================================================================

- Do NOT recompute or modify numeric values.
- Do NOT add new analytics or invent new fields.
- Produce UI-ready JSON only.
- Your must return Plotly.js figure JSON object that answers the user query.




USER QUERY
{user_query}

TABLE
{tables}


PLACEHOLDER RULE
Use only the raw column name as the placeholder.

Examples:
"x": "DATE"
"y": "oil_production"
"text": "producer"
"labels": "producer"
"values": "oil_production"


STYLE RULES
- Include a clear title.
- Include axis titles when the chart has x/y axes.
- Use hovertemplate when useful.
- Set config.responsive to true.
- Set config.displaylogo to false.



RETURN SHAPE
Return only JSON shaped like this:

{{
  "data": [...],
  "layout": {{
    
    "title": {{
      "text": "<chart title>"
    }},
        "colorway": ["#4A90E2", "#50E3C2"],

        "xaxis": {{
          "title": {{ "text": "X-axis title" }}
        }},

        "yaxis": {{
          "title": {{ "text": "Y-axis title" }}
        }},

  }},
  "config": {{
    "responsive": true,
    "displaylogo": false
  }}
}}
"""

In [491]:
VEGA_LITE_PROMPT = """
You are a Vega-Lite (v5) chart specification generator.

You are given:
- A user query.
- A table summary (table name, column names, and types).

===============================================================================
VISUALIZATION LOGIC
===============================================================================
1. Time Series: Use `mark: "line"` or `"area"`.
2. Grouped Comparisons: Use `mark: "bar"`. To group bars side-by-side, use the `xOffset` encoding.
3. Part-to-Whole: Use `mark: "arc"`.
4. Relationships: Use `mark: "point"` for scatter plots.

===============================================================================
TRANSFORM & DATA RULES
===============================================================================
- COLUMN CONCATENATION: If you need to combine columns (e.g., Year and Quarter), use a `calculate` transform:
  "transform": [{{"calculate": "datum.year + ' Q' + datum.quarter", "as": "period"}}]
- SORTING: Always ensure time-based axes are sorted chronologically using the `sort` property in the encoding.
- NO PLOTS WITH RAW DATA: Use the placeholder "TABLE_DATA" for the `values` key in the `data` block.
- DATA FORMAT: Assume the data is a flat list of dictionaries (one per row).

===============================================================================
STRICT VEGA-LITE JSON RULES
===============================================================================
- Enclose the JSON inside <plot> and </plot> tags.
- Use only valid Vega-Lite v5 properties.
- Do NOT invent column names.
- Always include a `title` and clear `axis` labels.

===============================================================================
INTERACTIVITY & ENHANCEMENT RULES (MANDATORY)
===============================================================================
- ZOOM & PAN: Always include the following params block at the top level:
  "params": [{{"name": "grid", "select": "interval", "bind": "scales"}}]
- TOOLTIPS: Always include a "tooltip" array in the encoding block containing all relevant columns.
- COLUMN CONCATENATION: If you need to combine columns (e.g., Year and Quarter), use a `calculate` transform:
  "transform": [{{"calculate": "datum.YEAR + ' Q' + datum.quarter", "as": "period"}}]
- SORTING: Always sort time-based axes chronologically using the `sort` property.


===============================================================================
OUTPUT FORMAT EXAMPLE
===============================================================================
Reasoning: [Briefly explain chart choice and any transforms used]
<plot>
{{
  "$schema": "https://github.io",
  "data": {{ "values": "TABLE_DATA" }},
  "transform": [ ... ],
  "mark": "bar",
  "encoding": {{
    "x": {{ "field": "period", "type": "nominal" }},
    "y": {{ "field": "total_water_injection_volume", "type": "quantitative" }},
    "xOffset": {{ "field": "SUBZONE" }},
    "color": {{ "field": "SUBZONE" }}
  }}
}}
</plot>

USER QUERY: {user_query}
TABLE: {tables}
"""


In [492]:

user_query = response['structured_response'].user_query
table_name = "quarterly_water_injection_by_subzone_year"
table = single_agent_data.catalog_snapshot( table_name ).derived_tables[0]
df = single_agent_data.get_table_as_df(table_name)
df

,YEAR,quarter,SUBZONE,total_water_injection_volume
0,2021,4,LW,2.457695e+06
1,2021,1,RW,2.061144e+06
2,2021,3,RW,1.993172e+06
3,2021,2,RW,2.139758e+06
4,2020,4,RW,1.740951e+06
5,2021,1,LW,4.148451e+06
6,2021,4,RW,1.191223e+06
7,2021,3,LW,4.069189e+06
8,2021,2,LW,4.189892e+06
9,2020,4,LW,3.554678e+06


In [493]:
import re 

p = TableResponseProcessor()

table_context = p.process( [(table,df)] )  

prompt = VEGA_LITE_PROMPT .format(
    tables = table_context,
    user_query = user_query
)




messages = [
    {'role':'system', 'content': prompt }
]
viz_response = llm.invoke(messages)



C:\Users\xteijeiro\AppData\Local\Temp\ipykernel_43340\917733957.py:174: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [494]:
viz_response.pretty_print()


================================== Ai Message ==================================

Reasoning: To visualize the total water injection volume for each quarter since 2018, split by subzone and year, a grouped bar chart is appropriate. Each bar will represent the total water injection volume for a specific quarter, grouped by subzone and year. A calculated field will combine the year and quarter into a single "period" field for the x-axis. The `xOffset` encoding will ensure the bars are grouped by subzone within each period. Tooltips will display all relevant details, and zoom/pan functionality will be included.

<plot>
{
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Total Water Injection Volume by Quarter, Year, and Subzone",
  "data": { "values": "TABLE_DATA" },
  "params": [
    {
      "name": "grid",
      "select": "interval",
      "bind": "scales"
    }
  ],
  "transform": [
    {
      "calculate": "datum.YEAR + ' Q' + datum.quarter",
      "as": "peri

In [495]:

import re
text = viz_response.content.strip()
text  = re.search(r'<plot>(.*?)</plot>', text, re.DOTALL)
##text  = re.search(r'```json(.*?)```', text, re.DOTALL)
text = text.group(1).strip()
  
#text = text.split("```json")[1].replace("```","")

print(text)


{
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Total Water Injection Volume by Quarter, Year, and Subzone",
  "data": { "values": "TABLE_DATA" },
  "params": [
    {
      "name": "grid",
      "select": "interval",
      "bind": "scales"
    }
  ],
  "transform": [
    {
      "calculate": "datum.YEAR + ' Q' + datum.quarter",
      "as": "period"
    }
  ],
  "mark": "bar",
  "encoding": {
    "x": {
      "field": "period",
      "type": "nominal",
      "title": "Year and Quarter",
      "sort": null
    },
    "xOffset": {
      "field": "SUBZONE",
      "title": "Subzone"
    },
    "y": {
      "field": "total_water_injection_volume",
      "type": "quantitative",
      "title": "Total Water Injection Volume"
    },
    "color": {
      "field": "SUBZONE",
      "type": "nominal",
      "title": "Subzone"
    },
    "tooltip": [
      { "field": "YEAR", "type": "ordinal", "title": "Year" },
      { "field": "quarter", "type": "ordinal", "title": "Qu

In [ ]:
import json
import altair as alt

# 1. Your JSON from the agent (as a string)

agent_output = """
{
  "$schema": "https://vega.github.io/schema/vega-lite/v6.json",
  "data": { "values": "TABLE_DATA" },

  "width": 800,
  "height": 420,

  "params": [
    {
      "name": "zoom_pan",
      "select": {
        "type": "interval"
      },
      "bind": "scales"
    },
    {
      "name": "subzone_toggle",
      "select": {
        "type": "point",
        "fields": ["SUBZONE"]
      },
      "bind": "legend"
    },
    {
      "name": "brush",
      "select": {
        "type": "interval",
        "encodings": ["x"]
      }
    },
    {
      "name": "hover",
      "select": {
        "type": "point",
        "on": "mouseover",
        "clear": "mouseout"
      }
    }
  ],

  "transform": [
    {
      "calculate": "toString(datum.YEAR) + ' Q' + toString(datum.quarter)",
      "as": "period"
    }
  ],

  "mark": {
    "type": "bar",
    "cursor": "pointer",
    "stroke": "black",
    "strokeWidth": 0.5
  },

  "encoding": {
    "x": {
      "field": "period",
      "type": "nominal",
      "title": "Quarter",
      "sort": null,
      "axis": {
        "labelAngle": 0
      }
    },

    "y": {
      "field": "total_water_injection_volume",
      "type": "quantitative",
      "title": "Total Water Injection Volume"
    },

    "xOffset": {
      "field": "SUBZONE",
      "type": "nominal"
    },

    "color": {
      "field": "SUBZONE",
      "type": "nominal",
      "title": "Subzone"
    },

    "opacity": {
      "condition": [
        {
          "param": "hover",
          "empty": false,
          "value": 1
        },
        {
          "param": "subzone_toggle",
          "value": 0.9
        },
        {
          "param": "brush",
          "empty": false,
          "value": 0.8
        }
      ],
      "value": 0.2
    },

    "tooltip": [
      {
        "field": "period",
        "type": "nominal",
        "title": "Quarter"
      },
      {
        "field": "SUBZONE",
        "type": "nominal",
        "title": "Subzone"
      },
      {
        "field": "total_water_injection_volume",
        "type": "quantitative",
        "title": "Water Injection Volume",
        "format": ",.2f"
      },
      {
        "field": "YEAR",
        "type": "ordinal",
        "title": "Year"
      },
      {
        "field": "quarter",
        "type": "ordinal",
        "title": "Quarter Number"
      }
    ]
  },

  "title": "Total Water Injection Volume by Quarter and Subzone"
}
"""


#agent_output = text 
# 2. Parse the string into a dictionary
chart_spec = json.loads(agent_output)

# 3. Swap the placeholder for your actual dataframe data
chart_spec['data']['values'] = df.to_dict(orient='records')

# 4. Display the chart
alt.Chart.from_dict(chart_spec).display()


alt.Chart(...)

In [502]:
chart = alt.Chart.from_dict(chart_spec).interactive()
chart.display()

chart.data


TypeError: unhashable type: 'ParameterName'

In [504]:
data_from_chart = pd.DataFrame(chart_spec['data']['values'])
data_from_chart

,YEAR,quarter,SUBZONE,total_water_injection_volume
0,2021,4,LW,2.457695e+06
1,2021,1,RW,2.061144e+06
2,2021,3,RW,1.993172e+06
3,2021,2,RW,2.139758e+06
4,2020,4,RW,1.740951e+06
5,2021,1,LW,4.148451e+06
6,2021,4,RW,1.191223e+06
7,2021,3,LW,4.069189e+06
8,2021,2,LW,4.189892e+06
9,2020,4,LW,3.554678e+06


In [478]:
%pip install altair


   ---------------------------------------- 0.0/797.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/797.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/797.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/797.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/797.0 kB ? eta -:--:--
   ------------------------ ------------- 524.3/797.0 kB 399.0 kB/s eta 0:00:01
   ------------------------ ------------- 524.3/797.0 kB 399.0 kB/s eta 0:00:01
   -------------------------------------  786.4/797.0 kB 419.4 kB/s eta 0:00:01
   ---------------------------------------- 797.0/797.0 kB 402.8 kB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [467]:

#text = viz_response.content.strip()
#text = re.sub(r"^```json\s*|\s*```$", "", text, flags=re.DOTALL)
figure = json.loads(text)


#figure['layout']["template"]="plotly_dark"


resolved_figure = resolve_plotly_figure(figure, df)
import plotly.io as pio
pio.show(resolved_figure)

In [445]:
df

,SUBZONE,year,quarter,total_water_injection_volume
0,LW,2020,4.0,3.554678e+06
1,LW,2021,3.0,4.069189e+06
2,LW,2021,2.0,4.189892e+06
3,RW,2020,4.0,1.740951e+06
4,RW,2021,3.0,1.993172e+06
5,RW,2021,2.0,2.139758e+06
6,LW,2021,4.0,2.457695e+06
7,RW,2021,1.0,2.061144e+06
8,RW,2021,4.0,1.191223e+06
9,LW,2021,1.0,4.148451e+06


In [126]:
list(pio.templates)

['ggplot2',
 'seaborn',
 'simple_white',
 'plotly',
 'plotly_white',
 'plotly_dark',
 'presentation',
 'xgridoff',
 'ygridoff',
 'gridon',
 'none']

In [20]:
c=""" 
CHART SELECTION RULES

First identify the main analytical intent of the user query:

1. Trend over an ordered variable
- Use a scatter trace with mode "lines+markers".
- Use this when the query asks for evolution, trend, history, change over time, monthly, yearly, daily, cumulative, forecast, or progression.
- x should be the temporal or ordered column.
- y should be the main quantitative column.
- Do not use pie charts for trends.

2. Category comparison / ranking
- Use a bar trace.
- Use this when the query asks to compare, rank, top N, bottom N, highest, lowest, by category, by well, by sector, by producer, by injector, etc.
- x should usually be the categorical column.
- y should be the quantitative column.
- If category labels are likely long, use a horizontal bar:
  - "type": "bar"
  - "orientation": "h"
  - x = quantitative column
  - y = categorical column

3. Part-to-whole / share
- Use a pie trace only when the query asks for share, percentage, fraction, contribution, mix, breakdown, distribution of a total, or proportion.
- Use pie only if the categorical column has low cardinality.
- For pie charts:
  - labels = categorical column
  - values = quantitative column
- Prefer a bar chart instead of pie if there are many categories, if ranking matters, or if categories do not represent parts of one total.

4. Numeric relationship / correlation
- Use a scatter trace with mode "markers".
- Use this when the query asks for relationship, correlation, vs, versus, dependency, association, crossplot, or compare two numeric variables row-by-row.
- x should be one quantitative column.
- y should be another quantitative column.
- Use text or hovertext for an identifier column if available.

5. Single value / one-row result
- If the table has one row and one main numeric value, use an indicator trace.
- If the table has one row and several numeric values, use a bar chart.
- If the user asks for a simple KPI/value, prefer an indicator.

6. Distribution of one numeric column
- If the query asks for distribution, histogram, frequency, spread, or variability, use a histogram trace.
- x should be the numeric column.

7. Multiple quantitative columns
- If the query asks to compare several metrics with the same unit, use multiple traces.
- If the metrics have different units, prefer separate y-axes only when necessary:
  - first metric uses "yaxis": "y"
  - second metric uses "yaxis": "y2"
  - layout must include yaxis and yaxis2
- Avoid dual axes unless the query clearly needs it.

8. Temporal + category + quantity
- If there is a temporal column, a categorical column with low/medium cardinality, and a quantitative column:
  - Use one scatter line trace only if the frontend will not split traces.
  - Put the category column in "text" or "hovertext".
  - Do not invent custom "split_by".
- If the input table is already wide, with one temporal column and several numeric series columns, create one trace per numeric series.

9. Heatmap / matrix-shaped tables
- Use heatmap only if the table clearly represents a matrix or has x-category, y-category, and numeric value columns.
- Use:
  - "type": "heatmap"
  - "x": categorical or temporal x column
  - "y": categorical y column
  - "z": numeric value column
- Do not use heatmap unless the table summary supports it.

General defaults:
- Prefer the simplest chart that answers the query.
- If the user explicitly asks for a chart type, follow it unless it conflicts with the data.
- If no clear chart type is implied:
  - temporal + quantitative -> line chart
  - categorical + quantitative -> bar chart
  - two quantitative columns -> scatter chart
  - categorical share of quantitative total -> pie chart
- Do not use Plotly transforms.
- Do not aggregate, calculate, sort, filter, or group in the figure JSON unless the result already exists in the provided table.

"""


len(c)/4,int(len(c.split()) * 1.3)

(965.0, 852)

In [ ]:
figure = json.loads(response.content)
import plotly.io as pio

pio.renderers.default = "notebook_connected"

fig = {
    "data": figure["data"],
    "layout": figure.get("layout", {}),
    "config": figure.get("config", {}),
}

pio.show(fig)

In [ ]:


presenter_agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=presenter_system_prompt,
        checkpointer=None,
    )

In [ ]:
r = response['structured_response']
tables = r.tables 
query = r.user_query 
table_names  = [ t.table_name for t in tables ]
descriptions = [t.description for t in tables ] 

print( query )
print( table_names[0] )
print( descriptions[0] )
single_agent_data.get_table_as_df( table_names[0])

#viz_prompt """
#
#"""

print('-----')
single_agent_data.catalog_snapshot('largest_yoy_injection_drop')#.derived_tables# get_tables_brief_description()

In [ ]:
r = response['structured_response']
query = r.user_query 
tables = r.tables 
table_names  = [ t.table_name for t in tables ]
descriptions = [t.description for t in tables ] 

kinds = [] 
for t in tables:
    if t.row_count < 2 : kinds.append( 'text' )

single_agent_data.get_table_as_df( table_names[0])
TableItemAgentResponse


In [ ]:
print(single_agnet_tools[0].func('largest_yoy_injection_drop_per_subzone'))#.catalog_snapshot()



In [ ]:
#single_agent_data.catalog_snapshot().derived_tables#'largest_yoy_injection_drop_per_subzone')

print( response['structured_response'].tables )


In [ ]:
table_name = response['structured_response'].tables[0].table_name
print(table_name )
single_agent_data.get_table_as_df( table_name )



In [ ]:
data._catalog.snapshot()#. catalog_snapshot().derived_tables# 'largest_yoy_injection_drop_per_subzone' )

# As a graph, tool or node. 

In [ ]:
from langgraph.graph import StateGraph, END

def executor_prompt_builder_from_structured_plan(
    idiom: str,
    idiom_rules: dict,
    structured_plan,
    user_query: str = "" 
) -> str:

    step_blocks = []
    for step in structured_plan.steps:

        source_tables = ", ".join(step.source_tables)
        reusable_tables = (
            ", ".join(step.reusable_tables)
            if step.reusable_tables
            else "None"
        )

        block = f"""
        Step {step.step_id}
        Target Table: {step.target_table}
        Source Tables: {source_tables}
        Reusable Tables: {reusable_tables}
        Logic: {step.logic}
        """.strip()

        step_blocks.append(block)
        
    formatted_plan = "\n\n".join(step_blocks)
    idiom_examples = "\n".join(
            [f"- {k}: {v}" for k, v in idiom_rules.items()]
        )
 
    # -----------------------------------------
    # Build final system prompt
    # -----------------------------------------
    prompt = system_prompt_sql_executor_template.format(
        idiom=idiom,
        idiom_examples=idiom_examples,
        plan=formatted_plan
    )

    if user_query:
        prompt = prompt + f"\n\n**USER QUERY**:\n{user_query}\n"
        
    return prompt

def planner_prompt_builder( tools:SmartDataTools )->str:
    #txt = data.catalog_snapshot()
    #txt = json.dumps( data.catalog_snapshot(), indent=3)
    txt = tools.catalog_snapshot()
    prompt = system_prompt_sql_planner_template.format(catalog=txt)

    return prompt 

class DataAnalystState(BaseModel):
    user_query: str
    refined_query: Optional[str] = None  
    plan: Optional[ExecutionPlan] = Field(
        default=None,
        description="Structured execution plan generated by planner",
    )

    execution_result: Optional[AgentTableResponse]  = Field(
        default=None,
        description="Executor output",
    )

    error: Optional[str] = Field(
        default=None,
        description="Execution or planning error",
    )

class DataAnalyst:

    def __init__(self, llm, tables_dict: None | Dict[str,pd.DataFrame] = None, 
                            known_table_models: None | Dict[str,TableCard] = None ):

        self._data  : SmartData #= SmartData()
        self._tools : List[StructuredTool] #= SmartDataTools( self._data ).get_tools()
        self._llm = llm 
        self._idiom = 'duckdb'
        self._graph = None 

    
        self._idiom_rules= {'date subtraction': "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
        'date truncation': "Use DATE_TRUNC('month', column).",
        'reserved keywords': 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
        'string concatenation': 'Use the || operator or CONCAT().',
        'boolean aggregation': 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.',
        'nested aggregates': 'Avoid nested aggregates—never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)',
        'cte helpers': 'Use CTEs to capture helper scalars (like current_year via MAX("DATE")) before performing group aggregations. For example:\n    WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)'
        }

        if tables_dict and known_table_models:
            self.initialize_from_known_tables( tables_dict, known_table_models)


    def get_result_as_dataframes(
        self,
        state: DataAnalystState | dict,
    ) -> Dict[str, pd.DataFrame]:
        """
        Retrieve all materialized result tables as DataFrames.

        Supports:
        - DataAnalystState
        - raw graph dict output
        """
        # -----------------------------------------------------
        # Normalize state
        # -----------------------------------------------------
        if isinstance(state, dict):
            state = DataAnalystState(**state)

        results = {}

        if state.execution_result is None:
            return results

        if not state.execution_result.tables:
            return results

        # -----------------------------------------------------
        # Load tables
        # -----------------------------------------------------
        for table_item in state.execution_result.tables:
            table_name = table_item.table_name

            try:
                results[table_name] = self._data.get_table_as_df(
                    table_name
                )

            except Exception as e:
                print(
                    f"Failed loading table '{table_name}': {str(e)}"
                )

        return results

    def initialize_from_known_tables( self, 
                                     tables_dict: Dict[str,pd.DataFrame], 
                                     known_table_models: Dict[str,TableCard] ):
   
        self._data = SmartData() 
        self._data.initialize_from_named_dataframes( tables_dict, known_table_models)
        self._smart_data_tools = SmartDataTools( self._data )#.get_tools()
        self._tools = self._smart_data_tools.get_tools() 
        
    def planner_node( self, state:DataAnalystState):
        
        # generate structured plan
        messages = [] 
        instruction = state.user_query
        planner_prompt = planner_prompt_builder(self._smart_data_tools)
        print(planner_prompt)
        llm = self._llm
        try: 
            messages = [{
                "role": "system",
                "content": planner_prompt
                },
                {
                "role": "user",
                "content": instruction
                }]

            # Structured planner
            structured_llm = llm.with_structured_output(ExecutionPlan)
            plan = structured_llm.invoke(messages)
            #print( plan )


            return state.model_copy(
                update={
                    "plan": plan,
                    #"error": None,
                })


            #return {
            #    #"catalog": catalog,
            #    "plan": plan#.dict() if hasattr(plan, "dict") else plan,
            #}
        
        except Exception as e:
            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })

  
    def execution_node( self, state:DataAnalystState):
  
        llm = self._llm
        plan = state.plan
        idiom= self._idiom
        idiom_rules = self._idiom_rules

        q = plan.user_query if plan.refined_query is None else  plan.refined_query

        executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan,q )

        print("\n" + "=" * 80)
        print("EXECUTOR PROMPT")
        print("=" * 80)
        print(executor_prompt)
        print("=" * 80 + "\n")
        

        agent = create_agent(
                model=llm,
                system_prompt=executor_prompt,
                tools=self._tools,
                response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )
        
        try:
            response = run_agent_stream_values(agent, {} ) 
        

            #print('******************************')
            #print(response)
            #print('******************************')
            


            #return {
            ##"catalog": catalog,
            #"execution_result": response#.dict() if hasattr(plan, "dict") else plan,
            #} 
        
            return state.model_copy(
            update={
                "execution_result": response['structured_response'],
                "error": None,
            })

        
        except Exception as e:
            print("STREAM FAILED:", str(e))


            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })



            #return {
            #"catalog": catalog,
            #"error": str(e)#.dict() if hasattr(plan, "dict") else plan,
            #} 
            
        
    # Conditional after planner
    def should_continue_node(self, state: DataAnalystState):
        if state.error:
            return END

        if state.plan is None:
            return END

        if hasattr(state.plan, "steps") and not state.plan.steps:
            return END

        return "execution"

    # ------------------------------------------------------------------
    # Graph builder
    # ------------------------------------------------------------------
    def as_graph(self):
        builder = StateGraph(DataAnalystState)

        builder.add_node("planner", self.planner_node)
        builder.add_node("execution", self.execution_node)

        builder.set_entry_point("planner")

        builder.add_conditional_edges(
            "planner",
            self.should_continue_node,
            {
                "execution": "execution",
                END: END,
            },
        )

        builder.add_edge("execution", END)

        self._graph = builder.compile()

        return self._graph

    # ------------------------------------------------------------------
    # Main runner
    # ------------------------------------------------------------------
    def run(self, user_query: str):
        """
        Execute full planner -> executor workflow.

        Parameters
        ----------
        user_query : str
            Natural language analytical request.

        Returns
        -------
        DataAnalystState
            Final validated workflow state.
        """
        if self._graph is None:
            self.as_graph()

        initial_state = DataAnalystState(
            user_query=user_query,
            refined_query=None,
            plan=None,
            execution_result=None,
            error=None,
        )

        result = self._graph.invoke(initial_state)
        return self.normalize_state(result)

        # LangGraph may return dict depending on version
        #if isinstance(result, dict):
        #    return DataAnalystState(**result)
        #return result

# ------------------------------------------------------------------
# Normalize output
# ------------------------------------------------------------------
    def normalize_state(self, result) -> DataAnalystState:
        """
        Convert graph output into validated DataAnalystState.
        """
        if isinstance(result, DataAnalystState):
            return result

        if isinstance(result, dict):
            return DataAnalystState(**result)

        raise TypeError(
            f"Unsupported graph output type: {type(result)}"
        )


    # ------------------------------------------------------------------
    # Direct graph invoke wrapper
    # ------------------------------------------------------------------
    def invoke_graph(self, user_query: str) -> DataAnalystState:
        """
        Direct graph call but always returns structured state.
        """
        if self._graph is None:
            self.as_graph()

        result = self._graph.invoke(
            DataAnalystState(
                user_query=user_query,
                refined_query=None,
                plan=None,
                execution_result=None,
                error=None,
            )
        )

        return self.normalize_state(result)


    # ------------------------------------------------------------------
    # Export as LangChain tool
    # ------------------------------------------------------------------
    def as_tool(self) -> StructuredTool:
        """
        Expose DataAnalyst as a reusable tool for other agents.
        """

        def _run_analysis(user_query: str) -> dict:
            state = self.run(user_query)
            return state 
        
            return {
                "plan": (
                    state.plan.model_dump()
                    if state.plan else None
                ),
                "execution_result": (
                    state.execution_result.model_dump()
                    if state.execution_result else None
                ),
                "error": state.error,
            }

        return StructuredTool.from_function(
            func=_run_analysis,
            name="data_analyst",
            description=(
                "Executes structured analytical workflows over known tabular datasets. "
                "Useful for SQL-style table generation, ranking, aggregations, "
                "time-series analysis, and derived table creation."
            ),
        )


In [ ]:
analyst = DataAnalyst( llm, df_dict, known_table_models )


In [ ]:

tool = analyst.as_tool()
print(tool)

# =========================================================
# Direct tool invocation
# =========================================================
tool_result = tool.invoke(
    {
        "user_query": (
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    }
)
print("TOOL RESULT:")
type(tool_result)
tool_result.execution_result.tables

In [ ]:
dfs = analyst.get_result_as_dataframes(tool_result)
print(dfs.keys())
first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)



In [ ]:

graph = analyst.as_graph() 
display( graph )

# =========================================================
# Direct graph invoke
# =========================================================
result = graph.invoke(
    DataAnalystState(
        user_query=(
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    )
)

# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)





In [ ]:
result
analyst.# .get_result_as_dataframes()#'injector_wells_ranked_by_water_injection')

In [ ]:
dir(analyst)

In [ ]:



# =========================================================
# Simple invoke test
# =========================================================
response = analyst.run(
    "Create two separate tables: "
    "one ranking injector wells by total water injection volume, "
    "and another ranking producer wells by total oil production volume."
)

print("FINAL RESPONSE:")
# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)

# End 


In [ ]:

llm = azure_llm_if()



planner_prompt = planner_prompt_builder(smart_data_tools)
print(planner_prompt)


In [ ]:
messages = [] 
instruction = query11


messages = [{
    "role": "system",
    "content": planner_prompt
    },
    {
    "role": "user",
    "content": instruction
    }]


plan = llm.invoke(messages).content

print( plan )

In [ ]:


idiom = 'duckdb'
idiom_rules= {'date subtraction': "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
 'date truncation': "Use DATE_TRUNC('month', column).",
 'reserved keywords': 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
 'string concatenation': 'Use the || operator or CONCAT().',
 'boolean aggregation': 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.',
 'nested aggregates': 'Avoid nested aggregates—never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)',
 'cte helpers': 'Use CTEs to capture helper scalars (like current_year via MAX("DATE")) before performing group aggregations. For example:\n    WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)'
}
executor_prompt = executor_prompt_builder(idiom, idiom_rules,plan)
agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        #response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 

In [ ]:
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))



# Add structure to the plan 


In [ ]:


# generate structured plan
messages = [] 
instruction = query11


messages = [{
    "role": "system",
    "content": planner_prompt
    },
    {
    "role": "user",
    "content": instruction
    }]

# Structured planner
structured_llm = llm.with_structured_output(ExecutionPlan)
plan = structured_llm.invoke(messages)
print( plan )






In [ ]:
def executor_prompt_builder_from_structured_plan(
    idiom: str,
    idiom_rules: dict,
    structured_plan,
    user_query: str = "" 
) -> str:

    step_blocks = []
    for step in structured_plan.steps:

        source_tables = ", ".join(step.source_tables)
        reusable_tables = (
            ", ".join(step.reusable_tables)
            if step.reusable_tables
            else "None"
        )

        block = f"""
        Step {step.step_id}
        Target Table: {step.target_table}
        Source Tables: {source_tables}
        Reusable Tables: {reusable_tables}
        Logic: {step.logic}
        """.strip()

        step_blocks.append(block)
        
    formatted_plan = "\n\n".join(step_blocks)
    idiom_examples = "\n".join(
            [f"- {k}: {v}" for k, v in idiom_rules.items()]
        )
 
    # -----------------------------------------
    # Build final system prompt
    # -----------------------------------------
    prompt = system_prompt_sql_executor_template.format(
        idiom=idiom,
        idiom_examples=idiom_examples,
        plan=formatted_plan
    )

    if user_query:
        prompt = prompt + f"\n\nThis is the user query:\n{user_query}"
        
    return prompt


executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan)


print(executor_prompt)



In [ ]:
executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan)


print(executor_prompt)

agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        #response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))





# Add structure to the output 

In [ ]:

llm = azure_llm_if()


In [ ]:
class PlanStep(BaseModel):
    step_id: int
    target_table: str
    source_tables: List[str]
    reusable_tables: List[str] = Field(default_factory=list)
    logic: str

class ExecutionPlan(BaseModel):
    #raw_query: str 
    user_query: str
    tables_needed: List[str] = Field( default=[],description="List of all the tables in the catalog that will be needed to answer the question")
    steps: List[PlanStep]



class TableItemAgentResponse(BaseModel):
    table_name: str = Field(description="Name of a materialized output table")
    description: str = Field(description="Brief summary of the table contents")
        
   
      
class AgentTableResponse(BaseModel):
    # Literal ensures the LLM chooses only these specific strings
    agent: Literal["analyst"] = Field(
        default="analyst", 
        description="The role of the agent. Always 'analyst'."
    )
    tables: List[str] = Field(default=[], description="Comma-separated list of table names")

    #text : Optional[str]  = Field(default=None, description="textual response")
    #tables: List[TableItemAgentResponse] = Field(default_factory=list, description="List of materialized output tables")
    
    

In [ ]:
agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))




# Prompt-chaining implementation 

In [ ]:
from typing import TypedDict
from typing import TypedDict, Optional, Dict, Any, Callable
from pydantic import BaseModel, Field
from typing import List
import sys, pathlib, json, pprint, pandas as pd 
from pathlib import Path
from pydantic import Field,BaseModel 
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

from runtime.v4.semantics.load_semantics import load_semantics
from runtime.v4.semantics.semantic_models import * 
from runtime.v4.analyst_agent.catalog import Catalog
from runtime.v4.analyst_agent.smart_data import SmartData
from runtime.v4.analyst_agent.smart_data_tools  import SmartDataTools


from langgraph.graph import StateGraph, END



class PlanStep(BaseModel):
    step_id: int
    target_table: str
    source_tables: List[str]
    reusable_tables: List[str] = Field(default_factory=list)
    logic: str

class ExecutionPlan(BaseModel):
    #raw_query: str 
    user_query: str
    refined_query: Optional[str] = None 
    tables_needed: List[str] = Field( default=[],description="List of all the tables in the catalog that will be needed to answer the question")
    steps: List[PlanStep]

class TableItemAgentResponse(BaseModel):
    table_name: str = Field(description="Name of a materialized output table")
    description: str = Field(description="Brief summary of the table contents")
        

class AgentTableResponse(BaseModel):
    # Literal ensures the LLM chooses only these specific strings
    agent: Literal["analyst"] = Field(
        default="analyst", 
        description="The role of the agent. Always 'analyst'."
    )
    user_query: str = Field( description='sanitized user query')
    tables: List[TableItemAgentResponse] = Field(default=[], description="Comma-separated list of table names")

    #text : Optional[str]  = Field(default=None, description="textual response")
    #tables: List[TableItemAgentResponse] = Field(default_factory=list, description="List of materialized output tables")
    
    

In [ ]:



def executor_prompt_builder_from_structured_plan(
    idiom: str,
    idiom_rules: dict,
    structured_plan,
    user_query: str = "" 
) -> str:

    step_blocks = []
    for step in structured_plan.steps:

        source_tables = ", ".join(step.source_tables)
        reusable_tables = (
            ", ".join(step.reusable_tables)
            if step.reusable_tables
            else "None"
        )

        block = f"""
        Step {step.step_id}
        Target Table: {step.target_table}
        Source Tables: {source_tables}
        Reusable Tables: {reusable_tables}
        Logic: {step.logic}
        """.strip()

        step_blocks.append(block)
        
    formatted_plan = "\n\n".join(step_blocks)
    idiom_examples = "\n".join(
            [f"- {k}: {v}" for k, v in idiom_rules.items()]
        )
 
    # -----------------------------------------
    # Build final system prompt
    # -----------------------------------------
    prompt = system_prompt_sql_executor_template.format(
        idiom=idiom,
        idiom_examples=idiom_examples,
        plan=formatted_plan
    )

    if user_query:
        prompt = prompt + f"\n\n**USER QUERY**:\n{user_query}\n"
        
    return prompt

def planner_prompt_builder( tools:SmartDataTools )->str:
    #txt = data.catalog_snapshot()
    #txt = json.dumps( data.catalog_snapshot(), indent=3)
    txt = tools.catalog_snapshot()
    prompt = system_prompt_sql_planner_template.format(catalog=txt)

    return prompt 



class DataAnalystState(BaseModel):
    user_query: str
    refined_query: Optional[str] = None  
    plan: Optional[ExecutionPlan] = Field(
        default=None,
        description="Structured execution plan generated by planner",
    )

    execution_result: Optional[AgentTableResponse]  = Field(
        default=None,
        description="Executor output",
    )

    error: Optional[str] = Field(
        default=None,
        description="Execution or planning error",
    )

class DataAnalyst:

    def __init__(self, llm, tables_dict: None | Dict[str,pd.DataFrame] = None, 
                            known_table_models: None | Dict[str,TableCard] = None ):

        self._data  : SmartData #= SmartData()
        self._tools : List[StructuredTool] #= SmartDataTools( self._data ).get_tools()
        self._llm = llm 
        self._idiom = 'duckdb'
        self._graph = None 

    
        self._idiom_rules= {'date subtraction': "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
        'date truncation': "Use DATE_TRUNC('month', column).",
        'reserved keywords': 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
        'string concatenation': 'Use the || operator or CONCAT().',
        'boolean aggregation': 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.',
        'nested aggregates': 'Avoid nested aggregates—never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)',
        'cte helpers': 'Use CTEs to capture helper scalars (like current_year via MAX("DATE")) before performing group aggregations. For example:\n    WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)'
        }

        if tables_dict and known_table_models:
            self.initialize_from_known_tables( tables_dict, known_table_models)


    def get_result_as_dataframes(
        self,
        state: DataAnalystState | dict,
    ) -> Dict[str, pd.DataFrame]:
        """
        Retrieve all materialized result tables as DataFrames.

        Supports:
        - DataAnalystState
        - raw graph dict output
        """
        # -----------------------------------------------------
        # Normalize state
        # -----------------------------------------------------
        if isinstance(state, dict):
            state = DataAnalystState(**state)

        results = {}

        if state.execution_result is None:
            return results

        if not state.execution_result.tables:
            return results

        # -----------------------------------------------------
        # Load tables
        # -----------------------------------------------------
        for table_item in state.execution_result.tables:
            table_name = table_item.table_name

            try:
                results[table_name] = self._data.get_table_as_df(
                    table_name
                )

            except Exception as e:
                print(
                    f"Failed loading table '{table_name}': {str(e)}"
                )

        return results

    def initialize_from_known_tables( self, 
                                     tables_dict: Dict[str,pd.DataFrame], 
                                     known_table_models: Dict[str,TableCard] ):
   
        self._data = SmartData() 
        self._data.initialize_from_named_dataframes( tables_dict, known_table_models)
        self._smart_data_tools = SmartDataTools( self._data )#.get_tools()
        self._tools = self._smart_data_tools.get_tools() 
        
    def planner_node( self, state:DataAnalystState):
        
        # generate structured plan
        messages = [] 
        instruction = state.user_query
        planner_prompt = planner_prompt_builder(self._smart_data_tools)
        print(planner_prompt)
        llm = self._llm
        try: 
            messages = [{
                "role": "system",
                "content": planner_prompt
                },
                {
                "role": "user",
                "content": instruction
                }]

            # Structured planner
            structured_llm = llm.with_structured_output(ExecutionPlan)
            plan = structured_llm.invoke(messages)
            #print( plan )


            return state.model_copy(
                update={
                    "plan": plan,
                    #"error": None,
                })


            #return {
            #    #"catalog": catalog,
            #    "plan": plan#.dict() if hasattr(plan, "dict") else plan,
            #}
        
        except Exception as e:
            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })

  
    def execution_node( self, state:DataAnalystState):
  
        llm = self._llm
        plan = state.plan
        idiom= self._idiom
        idiom_rules = self._idiom_rules

        q = plan.user_query if plan.refined_query is None else  plan.refined_query

        executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan,q )

        print("\n" + "=" * 80)
        print("EXECUTOR PROMPT")
        print("=" * 80)
        print(executor_prompt)
        print("=" * 80 + "\n")
        

        agent = create_agent(
                model=llm,
                system_prompt=executor_prompt,
                tools=self._tools,
                response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )
        
        try:
            response = run_agent_stream_values(agent, {} ) 
        

            #print('******************************')
            #print(response)
            #print('******************************')
            


            #return {
            ##"catalog": catalog,
            #"execution_result": response#.dict() if hasattr(plan, "dict") else plan,
            #} 
        
            return state.model_copy(
            update={
                "execution_result": response['structured_response'],
                "error": None,
            })

        
        except Exception as e:
            print("STREAM FAILED:", str(e))


            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })



            #return {
            #"catalog": catalog,
            #"error": str(e)#.dict() if hasattr(plan, "dict") else plan,
            #} 
            
        
    # Conditional after planner
    def should_continue_node(self, state: DataAnalystState):
        if state.error:
            return END

        if state.plan is None:
            return END

        if hasattr(state.plan, "steps") and not state.plan.steps:
            return END

        return "execution"

    # ------------------------------------------------------------------
    # Graph builder
    # ------------------------------------------------------------------
    def as_graph(self):
        builder = StateGraph(DataAnalystState)

        builder.add_node("planner", self.planner_node)
        builder.add_node("execution", self.execution_node)

        builder.set_entry_point("planner")

        builder.add_conditional_edges(
            "planner",
            self.should_continue_node,
            {
                "execution": "execution",
                END: END,
            },
        )

        builder.add_edge("execution", END)

        self._graph = builder.compile()

        return self._graph

    # ------------------------------------------------------------------
    # Main runner
    # ------------------------------------------------------------------
    def run(self, user_query: str):
        """
        Execute full planner -> executor workflow.

        Parameters
        ----------
        user_query : str
            Natural language analytical request.

        Returns
        -------
        DataAnalystState
            Final validated workflow state.
        """
        if self._graph is None:
            self.as_graph()

        initial_state = DataAnalystState(
            user_query=user_query,
            refined_query=None,
            plan=None,
            execution_result=None,
            error=None,
        )

        result = self._graph.invoke(initial_state)
        return self.normalize_state(result)

        # LangGraph may return dict depending on version
        #if isinstance(result, dict):
        #    return DataAnalystState(**result)
        #return result

# ------------------------------------------------------------------
# Normalize output
# ------------------------------------------------------------------
    def normalize_state(self, result) -> DataAnalystState:
        """
        Convert graph output into validated DataAnalystState.
        """
        if isinstance(result, DataAnalystState):
            return result

        if isinstance(result, dict):
            return DataAnalystState(**result)

        raise TypeError(
            f"Unsupported graph output type: {type(result)}"
        )


    # ------------------------------------------------------------------
    # Direct graph invoke wrapper
    # ------------------------------------------------------------------
    def invoke_graph(self, user_query: str) -> DataAnalystState:
        """
        Direct graph call but always returns structured state.
        """
        if self._graph is None:
            self.as_graph()

        result = self._graph.invoke(
            DataAnalystState(
                user_query=user_query,
                refined_query=None,
                plan=None,
                execution_result=None,
                error=None,
            )
        )

        return self.normalize_state(result)


    # ------------------------------------------------------------------
    # Export as LangChain tool
    # ------------------------------------------------------------------
    def as_tool(self) -> StructuredTool:
        """
        Expose DataAnalyst as a reusable tool for other agents.
        """

        def _run_analysis(user_query: str) -> dict:
            state = self.run(user_query)
            return state 
        
            return {
                "plan": (
                    state.plan.model_dump()
                    if state.plan else None
                ),
                "execution_result": (
                    state.execution_result.model_dump()
                    if state.execution_result else None
                ),
                "error": state.error,
            }

        return StructuredTool.from_function(
            func=_run_analysis,
            name="data_analyst",
            description=(
                "Executes structured analytical workflows over known tabular datasets. "
                "Useful for SQL-style table generation, ranking, aggregations, "
                "time-series analysis, and derived table creation."
            ),
        )




In [ ]:
analyst = DataAnalyst( llm, df_dict, known_table_models )

tool = analyst.as_tool()
print(tool)


# =========================================================
# Direct tool invocation
# =========================================================
tool_result = tool.invoke(
    {
        "user_query": (
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    }
)
print("TOOL RESULT:")
type(tool_result)
tool_result.execution_result.tables





graph = analyst.as_graph() 
display( graph )

# =========================================================
# Direct graph invoke
# =========================================================
result = graph.invoke(
    DataAnalystState(
        user_query=(
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    )
)

# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)




# =========================================================
# Simple invoke test
# =========================================================
response = analyst.run(
    "Create two separate tables: "
    "one ranking injector wells by total water injection volume, "
    "and another ranking producer wells by total oil production volume."
)

print("FINAL RESPONSE:")
#print(response)



In [ ]:

graph = analyst.as_graph() 
display( graph )

# Direct graph invoke
# =========================================================
result = graph.invoke(
    DataAnalystState(
        user_query=(
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    )
)

# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]

display(first_df)

In [ ]:
# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]

display(first_df)

In [ ]:
type(result)
result.keys()
type(result['execution_result'])
state = DataAnalystState(**result)
state

In [ ]:

# =========================================================
# Simple invoke test
# =========================================================
response = analyst.run(
    "Create two separate tables: "
    "one ranking injector wells by total water injection volume, "
    "and another ranking producer wells by total oil production volume."
)

print("FINAL RESPONSE:")
#print(response)



In [ ]:
type(response)

In [ ]:
# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(response)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]

display(first_df)


In [ ]:
if response.plan:
    print("\nPLAN:")
    print(response.plan.model_dump_json(indent=3))


if response.execution_result:#" in response:
    print("\nEXECUTION RESULT:")
    print(response.execution_result.model_dump_json(indent=3))

In [ ]:
response.execution_result.tables

In [ ]:

executor_prompt = executor_prompt_builder(idiom, idiom_rules,plan)

agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        #response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))


In [ ]:
tools

In [ ]:




class old:     

    def fdgdfgrun( self, user_query):

        llm = self._llm
        #planner
        planner_prompt = self._planner_prompt_builder()
        messages = [{
            "role": "system",
            "content": planner_prompt
            },
            {
            "role": "user",
            "content": user_query
            }]
        structured_llm = llm.with_structured_output(ExecutionPlan)

        plan = structured_llm.invoke(messages)
        print('Plan result')
        print( plan )
        
        executor_prompt = self._executor_prompt_builder(self._idiom, 
                                                        self._idiom_rules, 
                                                        plan )  
        executor_agent = create_agent(
                model=llm,
                system_prompt=executor_prompt,
                tools=self._tools,
                #response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )   
         
        

        #executor 
        try:
            response = run_agent_stream_values(executor_agent, {} ) 
            last_response = response 
        except Exception as e:
            print("STREAM FAILED:", str(e))





    def xxrun(self, user_query: str) -> Dict[str, Any]:

        initial_state: DataAnalystState = {
            "user_query": user_query,
            "plan": None,
            "execution_result": None,
            "error": None,
        }

        return self._graph.invoke(initial_state)    
    
    def planner_node(self,state: DataAnalystState) -> DataAnalystState:
        
        llm = self._llm
        
        # Structured planner
        structured_planner = llm.with_structured_output(ExecutionPlan)
        state["error"] = "" 
        try:
            planner_prompt = self._planner_prompt_builder()
            instruction = state['user_query']
            messages = [
                {
                    "role": "system",
                    "content": planner_prompt
                },
                {
                    "role": "user",
                    "content": instruction
                }
            ]

                
            
            plan = structured_planner.invoke(messages)
            
            state["plan"] = plan
            print("Plan produced")

        except Exception as e:
            state["error"] = f"Planner failed: {str(e)}"

        return state

    def executor_node(self,state: DataAnalystState) -> DataAnalystState:

        llm = self._llm
        if state.get("error"):
            return state

        try:
            plan = state["plan"]
            executor_system_prompt = self._executor_prompt_builder(self._idiom, 
                                                                   self._idiom_rules, 
                                                                   plan)
            
            executor_agent = create_agent(
                model=llm,
                system_prompt=executor_system_prompt,
                tools=self._tools,
                #response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )     
            
            
            result = executor_agent.invoke({
                "messages": [
                    {
                        "role": "system",
                        "content": executor_system_prompt
                    }

                ]
            })

            state["execution_result"] = result

        except Exception as e:
            state["execution_result"] = None 
            state["error"] = f"Executor failed: {str(e)}"

        return state







        return state 

    def _build_graph(self):

        builder = StateGraph(DataAnalystState)

        builder.add_node("planner", self.planner_node)
        builder.add_node("executor", self.executor_node)

        builder.set_entry_point("planner")

        builder.add_conditional_edges(
            "planner",
            self._should_continue,
            {
                "executor": "executor",
                "end": END,
            }
        )

        builder.add_edge("executor", END)

        return builder.compile()

    def _should_continue(
        self,
        state: DataAnalystState
    ) -> str:

        if state.get("error"):
            return "end"

        return "executor"
    
    def _planner_prompt_builder( self )->str:
        txt = self._data.catalog_snapshot()
        txt = json.dumps( self._data.catalog_snapshot(), indent=3)

        prompt = system_prompt_sql_planner_template.format(catalog=txt)

        return prompt 

    def _executor_prompt_builder( self, idiom, idiom_examples, plan )->str:
        prompt = system_prompt_sql_executor_template.format(idiom=idiom, idiom_examples=idiom_examples, plan=plan)
        return prompt 



        





In [ ]:
analyst.run( query9 )


In [ ]:
display(analyst._graph )


In [ ]:
for n,item in enumerate(queries):

    print("\n\n")
    print(120*'=')
    user_query = item[0]
    print( user_query, 30*' ', n  )
    print(120*'=')
    

    #user_query = "name the first tree wells in the last table"
    messages = {"messages": [{"role": "user", "content": user_query}]}
    
    try:
        response = run_agent_stream_values(agent, messages ) 
        last_response = response 
    except Exception as e:
        print("STREAM FAILED:", str(e))
    #response = agent.invoke(
    #messages,
    #config={"recursion_limit": 10},
    #)
    print(50*'=',sep="\n\n")
    break